# Bài tập: Phân loại Spam Email bằng Support Vector Machine (SVM)
* **Họ tên:** [Điền họ tên của bạn]
* **MSSV:** [Điền MSSV của bạn]


## 1. Thiết lập môi trường và Import thư viện


In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')


## 2. Tải và Khám phá dữ liệu (EDA)


In [ ]:
# Tải tập dữ liệu từ file local CSV
df = pd.read_csv('spam_ham_dataset.csv')

# Xem 5 dòng đầu
print('--- 5 DÒNG ĐẦU DỮ LIỆU ---')
display(df.head())

# Thông tin dữ liệu
print('\n--- THÔNG TIN DỮ LIỆU ---')
df.info()

# Thống kê nhãn
counts = df['label'].value_counts()
percent = df['label'].value_counts(normalize=True) * 100
print('\n--- THỐNG KÊ CHI TIẾT NHÃN ---')
print(f'Số lượng:\n{counts}')
print(f'\nTỉ lệ phần trăm:\n{percent}')

# Vẽ biểu đồ phân bố nhãn
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.countplot(data=df, x='label', palette='copper')
plt.title('Số lượng Mail Ham vs Spam')
plt.xlabel('Loại Email')
plt.ylabel('Số lượng')

plt.subplot(1, 2, 2)
plt.pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=140, colors=['#f39c12', '#d35400'])
plt.title('Tỉ lệ phần trăm dữ liệu')
plt.tight_layout()
plt.show()


## 3. Tiền xử lý dữ liệu (Preprocessing)


In [ ]:
# Xóa bỏ dữ liệu trống nếu có
df = df.dropna()

# Ánh xạ nhãn: spam -> 1, ham -> 0
df['label_num'] = df['label'].map({'spam': 1, 'ham': 0})

# Tách tập Test gốc (20%)
df_train_full, df_test = train_test_split(df, test_size=0.2, random_state=42)

# Lọc tập Train để có 35% Spam
df_spam_train = df_train_full[df_train_full['label_num'] == 1]
df_ham_train = df_train_full[df_train_full['label_num'] == 0]

n_spam = len(df_spam_train)
n_ham_needed = int((n_spam / 0.35) - n_spam)
df_ham_train_down = df_ham_train.sample(n=min(n_ham_needed, len(df_ham_train)), random_state=42)

# Gộp tập Train chính thức
df_train_final = pd.concat([df_spam_train, df_ham_train_down]).sample(frac=1, random_state=42)

X_train_raw = df_train_final['text'].astype(str).values
y_train = df_train_final['label_num'].values

# Vector hóa với TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=2500)
X_train = vectorizer.fit_transform(X_train_raw).toarray()
print(f'Kích thước dữ liệu huấn luyện sau vector hóa: {X_train.shape}')


## 4. Xây dựng và Huấn luyện mô hình


In [ ]:
class SVM_Scratch:
    def __init__(self, learning_rate=0.1, lambda_param=0.0001, iterations=1000):
        self.lr = learning_rate
        self.lambda_param = lambda_param
        self.iterations = iterations
        self.w = None
        self.b = None
        self.history = [] # Lưu lịch sử độ chính xác huấn luyện

    def fit(self, X, y):
        y_transformed = np.where(y <= 0, -1, 1)
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0

        for i in range(self.iterations):
            for idx, x_i in enumerate(X):
                condition = y_transformed[idx] * (np.dot(x_i, self.w) - self.b) >= 1
                if condition:
                    self.w -= self.lr * (2 * self.lambda_param * self.w)
                else:
                    self.w -= self.lr * (2 * self.lambda_param * self.w - np.dot(x_i, y_transformed[idx]))
                    self.b -= self.lr * y_transformed[idx]
            
            # Tính toán và lưu độ chính xác huấn luyện sau mỗi epoch
            if i % 10 == 0:
                approx = np.dot(X, self.w) - self.b
                y_pred = np.where(approx >= 0, 1, 0)
                acc = np.mean(y_pred == y)
                self.history.append(acc)

            if i % 100 == 0:
                print(f'Vòng lặp SVM {i}...')

    def predict(self, X):
        approx = np.dot(X, self.w) - self.b
        return np.where(approx >= 0, 1, 0)

# Khởi tạo và huấn luyện mô hình SVM với các tham số tối ưu cực hạn (learning_rate=0.1, lambda_param=0.0001)
model = SVM_Scratch(learning_rate=0.1, lambda_param=0.0001, iterations=500)
model.fit(X_train, y_train)
print('Huấn luyện SVM thành công!')

# Vẽ biểu đồ độ chính xác qua các vòng lặp
plt.figure(figsize=(10, 5))
plt.plot(range(0, len(model.history) * 10, 10), model.history, color='orange', label='Training Accuracy')
plt.title('Độ chính xác qua các vòng lặp (SVM Training Accuracy)')
plt.xlabel('Số vòng lặp (Iterations)')
plt.ylabel('Accuracy')
plt.grid(True)
plt.legend()
plt.show()

# Lưu mô hình thành file pkl
model_save = {
    'w': model.w,
    'b': model.b,
    'vocabulary': vectorizer.vocabulary_,
    'idf': vectorizer.idf_
}
filename = 'svm_scratch_model.pkl'
with open(filename, 'wb') as f:
    pickle.dump(model_save, f)
print(f'Đã lưu thành công tại: {filename}')


## 5. Dự đoán và Đánh giá kết quả


In [ ]:
PATH_MODEL_PKL = 'svm_scratch_model.pkl'
PATH_DATA_CSV = 'spam_ham_dataset.csv'

# Tải mô hình
with open(PATH_MODEL_PKL, 'rb') as f:
    model_data = pickle.load(f)

w = model_data['w']
b = model_data['b']
vocab = model_data['vocabulary']
saved_idf = model_data['idf']

# Đọc dữ liệu tập Test sạch
df = pd.read_csv(PATH_DATA_CSV)
df['label_num'] = df['label'].map({'spam': 1, 'ham': 0})
X_text = df['text'].astype(str).values
y = df['label_num'].values
_, X_test_raw, _, y_test = train_test_split(X_text, y, test_size=0.2, random_state=42)

# Tái tạo vectorizer
vectorizer = TfidfVectorizer(vocabulary=vocab)
vectorizer.idf_ = saved_idf
X_test = vectorizer.transform(X_test_raw).toarray()

def predict_svm(X, w, b):
    approx = np.dot(X, w) - b
    return np.where(approx >= 0, 1, 0)

y_pred = predict_svm(X_test, w, b)

# Đánh giá mô hình
acc = accuracy_score(y_test, y_pred)
print('='*50)
print(f'Độ chính xác SVM trên tập Test: {acc*100:.2f}%')
print('='*50)
print('\nBáo cáo phân loại chi tiết (Classification Report):')
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

# Vẽ biểu đồ đánh giá
plt.figure(figsize=(15, 6))
plt.subplot(1, 2, 1)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Dự Ham', 'Dự Spam'], yticklabels=['Thực Ham', 'Thực Spam'])
plt.title('Ma trận nhầm lẫn (SVM Scratch)')
plt.xlabel('Dự đoán')
plt.ylabel('Thực tế')

plt.subplot(1, 2, 2)
labels = ['Ham', 'Spam']
actual = [np.sum(y_test == 0), np.sum(y_test == 1)]
predicted = [np.sum(y_pred == 0), np.sum(y_pred == 1)]
x = np.arange(len(labels))
width = 0.35
plt.bar(x - width/2, actual, width, label='Thực tế', color='#f39c12')
plt.bar(x + width/2, predicted, width, label='Dự đoán', color='#d35400')
plt.xticks(x, labels)
plt.ylabel('Số lượng')
plt.title('So sánh Thực tế vs Dự đoán (SVM)')
plt.legend()
plt.tight_layout()
plt.show()


### Thử nghiệm dự đoán trên văn bản email mới


In [ ]:
new_emails = [
    "Congratulations! You've won a free iPhone. Click here to claim your prize!",
    "We detected a new sign-in attempt to your account. Please confirm your identity below if this wasn't you."
]

print('--- DỰ ĐOÁN TRÊN VĂN BẢN TÙY Ý ---')
for text in new_emails:
    X_new = vectorizer.transform([text]).toarray()
    pred = predict_svm(X_new, w, b)
    result = 'Spam' if pred[0] == 1 else 'Ham'
    print(f'Văn bản: "{text}"')
    print(f'-> Dự đoán: {result}\n')
